# Computational study: exact PEAC algorithms and prosumer behavior

The notebook runs three experiments designed around the paper's main claims:

1. paired runtime comparisons between Algorithm 1, Algorithm 2, and the matching Gurobi LP/MIP;
2. a 14-prosumer, four-season London/PVGIS energy-community case study;
3. bounded price-forecast errors, including an exploratory sales-enabled setting.

Start with `QUICK = True`. If every assertion passes, set it to `False`, restart the kernel, and run all cells for the paper results.

In [ ]:
from pathlib import Path
import platform
import subprocess
import sys
import tempfile

import matplotlib.pyplot as plt
import gurobipy as gp
import numpy as np
import pandas as pd
import yaml
from IPython.display import display

from microgrid.data import prepare_data
from microgrid.experiments import ALGORITHM_1, ALGORITHM_2, SELF_CONSUMPTION, run_case_study, run_robustness, run_runtime_benchmark
from microgrid.plots import forecast_robustness, microgrid_outcomes, performance_profiles, prosumer_behavior, runtime_scaling, save

ROOT = Path.cwd()
CONFIG = yaml.safe_load((ROOT / "config.yaml").read_text(encoding="utf-8"))
QUICK = True
RESULTS = ROOT / CONFIG["paths"]["results"]
FIGURES = ROOT / CONFIG["paths"]["figures"]
RESULTS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
ENVIRONMENT = {"python": sys.version.split()[0], "gurobi": ".".join(map(str, gp.gurobi.version())), "platform": platform.platform(), "processor": platform.processor()}
pd.Series(ENVIRONMENT).to_csv(RESULTS / "environment.csv", header=False)
print({"mode": "quick" if QUICK else "full", "root": str(ROOT), **ENVIRONMENT})

## 1. Exactness and implementation tests

In [ ]:
with tempfile.TemporaryDirectory(dir=RESULTS) as temporary:
    subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "--basetemp", temporary], check=True)

## 2. Empirical panel

The same households must pass every seasonal quality check. Assets are calibrated from annual load and remain fixed across seasons. Independent half-hour rounding is avoided by rounding cumulative energy.

In [ ]:
windows, sample_audit = prepare_data(CONFIG, QUICK)
sample_audit.to_csv(RESULTS / "sample_audit.csv", index=False)
selected = sample_audit[sample_audit.selected]
seasons = len(CONFIG["data"]["quick" if QUICK else "full"]["season_starts"])
panel = selected.groupby("household_id").season.nunique()
assert panel.eq(seasons).all()
assert len(windows) == len(panel) * seasons
assert all(window.demand_kwh.sum() > 0 for window in windows)
display(selected[["load_kwh", "relative_load", "positive_share"]].describe())
print(f"Balanced panel: {len(panel)} prosumers x {seasons} seasons = {len(windows)} weeks")

## 3. Runtime comparison

The controlled instances resample complete London/PVGIS days and cover load-only, load-dominant, balanced, and PV-dominant regimes. Prices retain the London tariff regimes and add autocorrelated, net-load-responsive innovations and rare spikes. General transaction bounds ensure that the flexible and block formulations are distinct and feasible. Algorithm 2 maintains its moving maxima with monotone deques in O(nC) time. Runtime includes Gurobi model construction; solver-only time is retained separately. A time limit is shown by an open triangle and counted as unsolved in performance profiles.

In [ ]:
runtime_results = run_runtime_benchmark(windows, CONFIG, QUICK)
specialized = runtime_results[~runtime_results.algorithm.str.startswith("Gurobi")]
gurobi = runtime_results[runtime_results.algorithm.str.startswith("Gurobi")]
assert specialized.status.eq("optimal").all()
assert gurobi.status.isin(["optimal", "time_limit"]).all()
assert runtime_results.runtime_seconds.gt(0).all()
solver_differences = gurobi.objective_difference.dropna().abs()
assert {"Gurobi LP", "Gurobi MIP"} <= set(gurobi[gurobi.status.eq("optimal")].algorithm)
assert solver_differences.max() < 1e-7
assert runtime_results.groupby("instance").algorithm.nunique().eq(4).all()
display(runtime_results.groupby(["algorithm", "profile_regime"]).runtime_seconds.describe())
times = runtime_results.pivot(index="instance", columns="algorithm", values="runtime_seconds")
speedups = pd.DataFrame({"flexible_speedup": times["Gurobi LP"] / times[ALGORITHM_1], "block_speedup": times["Gurobi MIP"] / times[ALGORITHM_2]})
speedups.describe().to_csv(RESULTS / "runtime_speedups.csv")
runtime_results.groupby(["algorithm", "profile_regime"]).runtime_seconds.describe().to_csv(RESULTS / "runtime_regimes.csv")
display(speedups.describe())

fig = runtime_scaling(runtime_results)
save(fig, "runtime_scaling", FIGURES)
plt.show()
fig = performance_profiles(runtime_results)
save(fig, "runtime_profiles", FIGURES)
plt.show()

## 4. Multi-prosumer microgrid case study

Each prosumer is a price taker and solves its own PEAC problem. Buying prices retain the observed London tariff and respond to aggregate scarcity. Three export-payment scenarios test price-design sensitivity. Exact schedules are compared with no storage and with myopic PV self-consumption. Participation scenarios combine optimized and myopic schedules before computing community outcomes.

In [ ]:
case_results, case_schedules, participation_results, primary_instances = run_case_study(windows, CONFIG, QUICK)
central = CONFIG["case_study"]["primary_price_scenario"]
primary = case_results[case_results.price_scenario.eq(central) & case_results.capacity_scale.eq(CONFIG["case_study"]["primary_capacity_scale"])]
paired = primary[primary.policy.isin([ALGORITHM_1, ALGORITHM_2])].pivot(index=["household_id", "season"], columns="policy", values="net_bill_gbp")
assert (paired[ALGORITHM_2] >= paired[ALGORITHM_1] - 1e-7).all()
block_premium_pence = 100 * (paired[ALGORITHM_2] - paired[ALGORITHM_1])
assert case_schedules.purchase_kwh.sum() > 0
assert case_schedules.soc_kwh.max() > 0
assert set(case_results.price_scenario) == set(CONFIG["case_study"]["price_scenarios"])
display(primary.groupby("policy")[["net_bill_gbp", "benefit_vs_self_consumption_percent", "self_sufficiency", "battery_throughput_kwh"]].median())
display(block_premium_pence.describe().rename("block premium (pence/week)"))
display(case_results[case_results.policy.eq(ALGORITHM_1)].groupby("price_scenario")[["net_bill_gbp", "benefit_vs_self_consumption_percent"]].median())
display(participation_results.groupby("participation_rate")[["community_bill_gbp", "import_kwh", "export_kwh", "peak_import_kw"]].mean())

fig = microgrid_outcomes(case_schedules, participation_results)
save(fig, "microgrid_outcomes", FIGURES)
plt.show()
fig = prosumer_behavior(case_results, CONFIG["case_study"]["primary_capacity_scale"], central)
save(fig, "prosumer_behavior", FIGURES)
plt.show()

## 5. Price-forecast robustness

A demand-and-storage purchase-only scenario is compared with the paper's multiplicative realized-cost envelope; the sales-enabled scenario retains the empirical prosumer profile. Common autocorrelated error paths are scaled across error levels. With sales, optimum bills may be negative, so the meaningful outcome is additive regret divided by gross import cost. No multiplicative sales guarantee is asserted.

In [ ]:
robustness_results = run_robustness(primary_instances, CONFIG, QUICK)
purchase_only = robustness_results[~robustness_results.sales_allowed]
assert set(robustness_results.scenario) == {"purchase_only", "sales_enabled"}
assert (purchase_only.realized_ratio <= purchase_only.guarantee + 1e-8).all()
assert robustness_results.regret_gbp.ge(0).all()
sales = robustness_results[robustness_results.sales_allowed]
display(sales.groupby("epsilon")[["regret_gbp", "normalized_regret_percent", "schedule_change_kwh"]].quantile([0.5, 0.95]))
print(f"Sales cases with negative true bills: {100 * sales.negative_true_bill.mean():.1f}%")

fig = forecast_robustness(robustness_results)
save(fig, "forecast_robustness", FIGURES)
plt.show()

## 6. Paper tables and completion audit

In [ ]:
case_results.groupby(["price_scenario", "policy", "capacity_scale"])[["net_bill_gbp", "benefit_vs_no_storage_percent", "benefit_vs_self_consumption_percent", "self_sufficiency"]].describe().to_csv(RESULTS / "case_summary.csv")
robustness_results.groupby(["scenario", "epsilon"])[["regret_gbp", "normalized_regret_percent", "schedule_change_kwh"]].describe().to_csv(RESULTS / "robustness_summary.csv")
print(f"All checks passed. Results: {RESULTS.resolve()}")
print(f"Figures: {FIGURES.resolve()}")